# 0. Reestruturação da planilha bruta — Thermal Predictor

O arquivo `assets/data/dataset.xlsx` (aba **"Option 1 - Last Cases"**) está em formato "largo": cada **Run** é uma coluna e cada **termopar (TC)** é uma linha, com as specs de cada run (Reason, Blower Direction, Housing, etc.) empilhadas em linhas de cabeçalho acima da tabela de temperaturas.

Este notebook faz **apenas a reestruturação do formato** (wide → long): nenhum dado é calculado, filtrado ou alterado — só redistribuído. O resultado é um dataframe com uma linha por combinação (Run × TC), contendo as specs do run e do TC como colunas, salvo em `data/dataset_long.csv`.

## 1. Carregar a planilha bruta

In [1]:
import pandas as pd

raw = pd.read_excel('../assets/data/dataset.xlsx', sheet_name='Option 1 - Last Cases', header=None)
raw.shape

(93, 43)

## 2. Specs por Run

As linhas 0–8 (colunas 4 em diante) trazem, uma spec por linha, os metadados de cada run: rótulo, Reason, Blower Direction, Housing, Board Position, Run Case, Mesh Duct, Losses e Deflector Hole.

In [2]:
RUN_SPEC_ROWS = {
    0: 'run_label',
    1: 'reason',
    2: 'blower_direction',
    3: 'housing',
    4: 'board_position',
    5: 'run_case',
    6: 'mesh_duct',
    7: 'losses',
    8: 'deflector_hole',
}
RUN_COL_START = 4

run_block = raw.loc[list(RUN_SPEC_ROWS.keys()), RUN_COL_START:]
run_meta = run_block.T
run_meta.columns = [RUN_SPEC_ROWS[i] for i in run_block.index]
run_meta.index.name = 'col_pos'
run_meta = run_meta.reset_index()
run_meta.head()

,col_pos,run_label,reason,blower_direction,housing,board_position,run_case,mesh_duct,losses,deflector_hole
0,4,Run 19,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
1,5,Run 20,New Baseline - Simplified Tray,To Front,Open,Center,Broil Qualitative,Correct Level,NaN,Closed
2,6,Run 21,New Baseline - Hole Tray,To Front,Open,Center,Broil C2,Correct Level,NaN,Closed
3,7,Run 22,New Baseline - Real Tray,To Front,Open,Center,Broil Qualitative,Correct Level,NaN,Closed
4,8,Run 22,New Baseline - Hole Tray,To Front,Open,Center,Broil Qualitative,Correct Level,NaN,Closed


### 2.1 Desambiguar rótulos de run duplicados

A planilha bruta reusa o mesmo rótulo de run em algumas colunas (ex: "Run 22" aparece 2x). Para gerar um `run_id` único por coluna, mantemos o rótulo original na primeira ocorrência e sufixamos as repetições: `"Run 22"`, `"Run 22 (2)"`.

In [3]:
dup_count = run_meta.groupby('run_label').cumcount()
run_meta['run_id'] = [
    label if dup_count[i] == 0 else f"{label} ({dup_count[i] + 1})"
    for i, label in enumerate(run_meta['run_label'])
]

print(f"{run_meta['run_id'].nunique()} run_id únicos de {len(run_meta)} runs")
run_meta[run_meta['run_label'].duplicated(keep=False)][['col_pos', 'run_label', 'run_id']]

39 run_id únicos de 39 runs


,col_pos,run_label,run_id
3,7,Run 22,Run 22
4,8,Run 22,Run 22 (2)
20,24,Run 43,Run 43
21,25,Run 44,Run 44
22,26,Run 45,Run 45
23,27,Run 44,Run 44 (2)
24,28,Run 43,Run 43 (2)
25,29,Run 44,Run 44 (3)
26,30,Run 45,Run 45 (2)
35,39,Run 73,Run 73


## 3. Specs por TC (termopar)

A partir da linha 9, colunas 0–3 trazem `TC Number`, `TC Name`, `Key Area` e `Active Limit` para cada termopar. Intercaladas nessas linhas existem **linhas de cabeçalho de seção** (ex: "Mainback Components", "IPC Components") que identificamos como linhas com no máximo 2 valores preenchidos. Elas servem apenas para preencher (`ffill`) os poucos `Key Area` vazios dentro de cada bloco — a coluna `Key Area` já vem preenchida na maior parte das linhas.

In [4]:
section_rows = [
    i for i in range(9, raw.shape[0])
    if raw.iloc[i].notna().sum() <= 2 and pd.notna(raw.iloc[i, 0])
]
print("Linhas de cabeçalho de seção:", section_rows)
list(raw.iloc[section_rows, 0])

Linhas de cabeçalho de seção: [9, 21, 30, 38, 44, 82]


['Mainback Components ',
 'Coil Temperatures ',
 'Main Ventilation System',
 'Cooktop/IPC Ventilation System',
 'IPC Components',
 'HMI / Knobs / Console Region ']

In [5]:
tc_rows = [i for i in range(9, raw.shape[0]) if i not in section_rows]

tc_meta = raw.loc[tc_rows, [0, 1, 2, 3]].copy()
tc_meta.columns = ['tc_number', 'tc_name', 'key_area', 'active_limit']

bounds = section_rows + [raw.shape[0]]
for b in range(len(section_rows)):
    start, end = section_rows[b] + 1, bounds[b + 1]
    block_idx = [i for i in tc_meta.index if start <= i < end]
    tc_meta.loc[block_idx, 'key_area'] = tc_meta.loc[block_idx, 'key_area'].ffill()

print("TCs:", tc_meta.shape[0], "| Key Area vazio após ffill:", tc_meta['key_area'].isna().sum())
tc_meta.head()

TCs: 78 | Key Area vazio após ffill: 0


,tc_number,tc_name,key_area,active_limit
10,4,"Door Lock Motor Ambient 1/2"" Below",Mainback,155
11,5,Door Lock Microswitch Body,Mainback,136
12,6,Convect Motor Winding,Mainback,165
13,7,Hot Surface Indicator Body,Mainback,85
14,8,Power On Indicator Body,Mainback,85


## 4. Melt: de largo (Run × TC em matriz) para longo (uma linha por Run × TC)

In [6]:
temps = raw.loc[tc_rows, RUN_COL_START:].copy()
temps.columns = run_meta['run_id'].values
temps.index = tc_meta.index

long_df = temps.join(tc_meta).melt(
    id_vars=['tc_number', 'tc_name', 'key_area', 'active_limit'],
    var_name='run_id',
    value_name='temp',
)

run_spec_cols = ['run_id', 'reason', 'blower_direction', 'housing', 'board_position',
                 'run_case', 'mesh_duct', 'losses', 'deflector_hole']
long_df = long_df.merge(run_meta[run_spec_cols], on='run_id', how='left')

# uma célula em branco na planilha original = TC não medido nesse run (não é dado a reestruturar)
long_df = long_df.dropna(subset=['temp']).reset_index(drop=True)

long_df = long_df[[
    'run_id', 'tc_number', 'tc_name', 'key_area', 'active_limit', 'temp',
    'reason', 'blower_direction', 'housing', 'board_position',
    'run_case', 'mesh_duct', 'losses', 'deflector_hole',
]]
long_df.shape

(2862, 14)

## 5. Conferir o resultado

In [7]:
long_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2862 entries, 0 to 2861
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   run_id            2862 non-null   str   
 1   tc_number         2706 non-null   object
 2   tc_name           2862 non-null   str   
 3   key_area          2862 non-null   str   
 4   active_limit      2862 non-null   object
 5   temp              2862 non-null   object
 6   reason            2862 non-null   object
 7   blower_direction  2862 non-null   object
 8   housing           2862 non-null   object
 9   board_position    2862 non-null   object
 10  run_case          2862 non-null   object
 11  mesh_duct         2862 non-null   object
 12  losses            2444 non-null   object
 13  deflector_hole    2862 non-null   object
dtypes: object(11), str(3)
memory usage: 313.2+ KB


In [9]:
long_df.shape

(2862, 14)

In [8]:
long_df.head(20)

,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,mesh_duct,losses,deflector_hole
0,Run 19,4,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,82.11,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
1,Run 19,5,Door Lock Microswitch Body,Mainback,136,88.25,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
2,Run 19,6,Convect Motor Winding,Mainback,165,103.87,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
3,Run 19,7,Hot Surface Indicator Body,Mainback,85,43.42,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
4,Run 19,8,Power On Indicator Body,Mainback,85,40.44,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
5,Run 19,9,Oven RTD Connector Body,Mainback,150,201.57,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
6,Run 19,#REF!,Storage Drawer (Metal Above),Mainback,150,96.72,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
7,Run 19,#REF!,"L2, Wire, Power Cord",Mainback,60,77.33,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
8,Run 19,#REF!,Ambient Below Terminal Block,Mainback,60,73.91,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open
9,Run 19,#REF!,ACU Ambient (Hill Only) Top,Mainback,85,90.86,New Baseline,To Back,Open,Center,Broil Qualitative,Correct Level,NaN,Open


In [9]:
# runs presentes e conferência de que os duplicados foram desambiguados
print(long_df['run_id'].nunique(), "runs distintos")
assert long_df.groupby('run_id').ngroups == run_meta['run_id'].nunique()
sorted(long_df['run_id'].unique())

39 runs distintos


['Run 19',
 'Run 20',
 'Run 21',
 'Run 22',
 'Run 22 (2)',
 'Run 23',
 'Run 24 - Rerunned',
 'Run 25',
 'Run 26',
 'Run 27',
 'Run 28',
 'Run 29',
 'Run 30 ',
 'Run 31 ',
 'Run 32',
 'Run 33',
 'Run 34',
 'Run 35',
 'Run 38',
 'Run 41',
 'Run 43',
 'Run 43 (2)',
 'Run 44',
 'Run 44 (2)',
 'Run 44 (3)',
 'Run 45',
 'Run 45 (2)',
 'Run 46',
 'Run 47',
 'Run 48',
 'Run 57',
 'Run 60',
 'Run 61',
 'Run 62',
 'Run 63',
 'Run 73',
 'Run 73 (2)',
 'Run 74',
 'Run 74 (2)']

## 6. Salvar em `data/dataset_long.csv`

In [10]:
long_df.to_csv('../data/dataset_long.csv', index=False)
print("Salvo em data/dataset_long.csv:", long_df.shape)

Salvo em data/dataset_long.csv: (2862, 14)
